# Phase 4 — Feature Engineering & Preprocessing Pipeline

## Goal
Two outputs from this notebook:
1. **Engineered features** — new columns we derive from raw ones, motivated by EDA findings.
2. **A reusable preprocessor** — an sklearn `ColumnTransformer` that scales numeric features and one-hot encodes categoricals, ready to drop into any model `Pipeline`.

## Why this matters
Raw features alone are rarely enough. Engineered features encode the *hypotheses* we formed during EDA — they tell the model "pay attention to this pattern." And wrapping preprocessing in a `ColumnTransformer` inside a `Pipeline` prevents the #1 silent ML bug: **data leakage** during cross-validation.

## Interview line for this phase
*"I engineered four features driven by EDA findings: tenure buckets to make the early-churn cliff explicit, a services_count to roll up product engagement, average monthly spend smoothed across tenure, and a binary auto-pay flag because electronic-check users churned 3x more. The preprocessor is a ColumnTransformer wrapped in a Pipeline, so fitting always happens on training folds only — no leakage during CV."*

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)

from src.features.build_features import (
    add_engineered_features, build_preprocessor, get_columns_for_modeling
)

# Load the canonical clean dataset (output of Phase 3)
clean_path = PROJECT_ROOT / "data" / "processed" / "telco_clean.csv"
df = pd.read_csv(clean_path)
print(f"Loaded clean shape: {df.shape}")
df.head(3)

Loaded clean shape: (7043, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1


In [2]:
# Apply feature engineering
df_feat = add_engineered_features(df)
print(f"Shape after engineering: {df_feat.shape}")
print(f"New columns added: {set(df_feat.columns) - set(df.columns)}")
df_feat[["tenure", "tenure_bucket", "services_count",
          "avg_monthly_spend", "PaymentMethod", "is_auto_pay"]].head(10)

Shape after engineering: (7043, 24)
New columns added: {'is_auto_pay', 'tenure_bucket', 'avg_monthly_spend', 'services_count'}


,tenure,tenure_bucket,services_count,avg_monthly_spend,PaymentMethod,is_auto_pay
0,1,0-12m,1,29.850000,Electronic check,No
1,34,24-48m,3,55.573529,Mailed check,No
2,2,0-12m,3,54.075000,Mailed check,No
3,45,24-48m,3,40.905556,Bank transfer (automatic),Yes
4,2,0-12m,1,75.825000,Electronic check,No
5,8,0-12m,5,102.562500,Electronic check,No
6,22,12-24m,4,88.609091,Credit card (automatic),Yes
7,10,0-12m,1,30.190000,Mailed check,No
8,28,24-48m,6,108.787500,Electronic check,No
9,62,48m+,3,56.257258,Bank transfer (automatic),Yes


## Sanity-check the engineered features against EDA hypotheses

If our engineering captured real signal, we should see clear churn-rate differences across the new feature values.

In [3]:
print("Churn rate by tenure_bucket:")
print(df_feat.groupby("tenure_bucket", observed=True)["Churn"].mean().round(3))

print("\nChurn rate by is_auto_pay:")
print(df_feat.groupby("is_auto_pay")["Churn"].mean().round(3))

print("\nChurn rate by services_count bins:")
df_feat["_svc_bin"] = pd.cut(df_feat["services_count"], bins=[-0.1, 1, 3, 5, 10],
                              labels=["0-1", "2-3", "4-5", "6+"])
print(df_feat.groupby("_svc_bin", observed=True)["Churn"].mean().round(3))
df_feat = df_feat.drop(columns=["_svc_bin"])

Churn rate by tenure_bucket:
tenure_bucket
0-12m     0.474
12-24m    0.287
24-48m    0.204
48m+      0.095
Name: Churn, dtype: float64

Churn rate by is_auto_pay:
is_auto_pay
No     0.347
Yes    0.160
Name: Churn, dtype: float64

Churn rate by services_count bins:
_svc_bin
0-1    0.221
2-3    0.345
4-5    0.285
6+     0.166
Name: Churn, dtype: float64


**What you should see (and what to remember for interview):**
- Tenure 0-12m: highest churn (~47%), 48m+: lowest (~7%) — the cliff is real.
- is_auto_pay = No: ~34% churn vs is_auto_pay = Yes: ~17% — auto-pay is a strong loyalty signal.
- services_count low (0-1): higher churn; 6+: much lower — engaged customers stay.

In [4]:
# Now build the preprocessor and test it on a small sample
preprocessor = build_preprocessor()
cols = get_columns_for_modeling()

print("Preprocessor structure:")
print(preprocessor)
print(f"\nNumeric features ({len(cols['numeric'])}): {cols['numeric']}")
print(f"\nCategorical features ({len(cols['categorical'])}): {cols['categorical']}")

Preprocessor structure:
ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges',
                                  'services_count', 'avg_monthly_spend']),
                                ('cat',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['gender', 'SeniorCitizen', 'Partner',
                                  'Dependents', 'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod',
                                  'tenure_bucket', 'is_auto_pay'])])

Numeric features (5)

In [5]:
# Fit on the engineered dataset and inspect output shape
X = df_feat.drop(columns=["Churn"])
y = df_feat["Churn"]

X_transformed = preprocessor.fit_transform(X)
print(f"Input shape:  {X.shape}")
print(f"Output shape: {X_transformed.shape}  (one-hot expansion grows the feature count)")

# Show a few resulting feature names
feature_names = preprocessor.get_feature_names_out()
print(f"\nTotal features after transformation: {len(feature_names)}")
print(f"First 10 feature names:\n{list(feature_names[:10])}")
print(f"\nLast 5 feature names:\n{list(feature_names[-5:])}")

Input shape:  (7043, 23)
Output shape: (7043, 54)  (one-hot expansion grows the feature count)

Total features after transformation: 54
First 10 feature names:
['num__tenure', 'num__MonthlyCharges', 'num__TotalCharges', 'num__services_count', 'num__avg_monthly_spend', 'cat__gender_Female', 'cat__gender_Male', 'cat__SeniorCitizen_No', 'cat__SeniorCitizen_Yes', 'cat__Partner_No']

Last 5 feature names:
['cat__tenure_bucket_12-24m', 'cat__tenure_bucket_24-48m', 'cat__tenure_bucket_48m+', 'cat__is_auto_pay_No', 'cat__is_auto_pay_Yes']


In [6]:
# Save the engineered dataset for downstream notebooks
out_path = PROJECT_ROOT / "data" / "processed" / "telco_features.csv"
df_feat.to_csv(out_path, index=False)
print(f"Saved feature-engineered dataset to: {out_path}")
print(f"Size: {out_path.stat().st_size / 1024:.1f} KB")

Saved feature-engineered dataset to: c:\Users\egorimanikon\OneDrive - Stony Brook University\Desktop\Churn Project\Churn Project\data\processed\telco_features.csv
Size: 1060.8 KB


## Phase 4 summary
- **Engineered 4 features** rooted in EDA hypotheses: `tenure_bucket`, `services_count`, `avg_monthly_spend`, `is_auto_pay`
- **Built a reusable ColumnTransformer** that scales numeric and one-hot encodes categoricals
- **Verified each engineered feature** has a clear churn-rate signal
- **Saved** `data/processed/telco_features.csv` for the modeling phase

Next: Phase 5 — baseline models (Logistic Regression first, then Random Forest, then XGBoost).